In [9]:
import logging
import tempfile
from pathlib import Path

from dotenv import load_dotenv

import databao.agent as bao
from databao.agent.configs.agent import AgentConfig
from databao.agent.databases import DuckDBConnectionConfig
from databao.agent.executors import ClaudeAgentExecutor
from databao.agent.executors.query_expansion import QueryExpansionConfig

load_dotenv()

logging.basicConfig(level=logging.INFO)

EXAMPLES_DIR = Path.cwd()
# NOTE: (@gas) in order to build the context with DCE,
# dbt project should be "initialized", e.g. with `dbt run`;
# the demo project is taken from the Spider-2-dbt dataset
DBT_PROJ_PATH = EXAMPLES_DIR / "shopify002"
DB_PATH = DBT_PROJ_PATH / "shopify.duckdb"
DOMAIN_PATH = "/Users/andrei.gasparian/Documents/databao-project/databao/domains/root"

In [2]:
llm_config = bao.LLMConfig(name="gpt-5", temperature=0)
agent_config = AgentConfig(recursion_limit=100, parallel_tool_calls=True)

In [3]:
domain_ctx = bao.domain(project_dir=DOMAIN_PATH)

In [4]:
agent = bao.agent(
    domain=domain_ctx,
    name="demo-dbt-executor",
    llm_config=llm_config,
    agent_config=agent_config,
    data_executor=ClaudeAgentExecutor(),
)

In [5]:
thread = agent.thread(stream_ask=True)

In [6]:
thread.ask(
    "What is our refund rate by month?",
    metadata={"source": "shopify_dbt"},
)

INFO:claude_agent_sdk._internal.transport.subprocess_cli:Using bundled Claude Code CLI: /Users/andrei.gasparian/Documents/databao-agent/.venv/lib/python3.12/site-packages/claude_agent_sdk/_bundled/claude


,month,total_orders,refunded_orders,refunded_amount,gross_sales,refund_rate_order_pct,refund_rate_amount_pct
0,2021-03-01,110,2.0,54.0,15161.07,1.82,0.36
1,2021-02-01,280,2.0,84.2,46161.89,0.71,0.18
2,2021-01-01,310,2.0,108.0,45482.19,0.65,0.24
3,2020-12-01,310,2.0,94.0,50155.70,0.65,0.19
4,2020-11-01,300,2.0,94.0,43816.56,0.67,0.21
5,2020-10-01,310,2.0,126.4,49248.23,0.65,0.26
6,2020-09-01,193,2.0,68.0,29433.19,1.04,0.23


In [7]:
print("\n=== TEXT ===\n")
print(thread.text())


=== TEXT ===

Here's your **refund rate by month** across 7 months of data (Sep 2020 – Mar 2021):

**Key takeaways:**
- **Order-based refund rate** (% of orders refunded) ranges from **0.65% to 1.82%**, with March 2021 being the highest.
- **Amount-based refund rate** (refunded $ / gross sales) is consistently low, ranging from **0.18% to 0.36%** — meaning the refunded orders tend to be relatively small in value.
- Each month saw exactly **2 refunded orders**, so the rate fluctuation is driven by order volume — higher volume months (Jan–Feb 2021, ~300 orders) show lower rates.

Two rate definitions are tracked:
| Metric | Definition |
|---|---|
| `refund_rate_order_pct` | % of orders that had any refund |
| `refund_rate_amount_pct` | Refunded amount as % of gross sales |
